In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [5]:
data = pd.read_csv('Churn_Modelling.csv')
## Preprocessing the data
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1) 

## Encoding the categorical data
labelencoder = LabelEncoder()
data['Gender'] = labelencoder.fit_transform(data['Gender'])
#data.head()

onehotencoder = OneHotEncoder(handle_unknown='ignore')
geography = onehotencoder.fit_transform(data['Geography'].values.reshape(-1,1)).toarray()
geography_df = pd.DataFrame(geography, columns=onehotencoder.get_feature_names_out(['Geography']))
#geography_df.head()

data =pd.concat([data.drop('Geography',axis=1),geography_df],axis=1)
#data.head()

X= data.drop('Exited',axis=1)
y= data['Exited']

In [6]:
### Train test split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
### Save all the pickle files for later use
with open('labelencoder.pkl','wb') as file:
    pickle.dump(labelencoder,file)

with open('onehotencoder.pkl','wb') as file:
    pickle.dump(onehotencoder,file)

with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)


In [10]:
### Define a function to create the model and try different hyperparameters(KerasClassifier)

def create_model(neurons=32,layer=1):   
    model = Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layer-1):
        model.add(Dense(neurons,activation='relu'))
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
    
    return model

In [17]:
### Create a KerasClassifier
model = KerasClassifier(model=create_model, verbose=0)

In [18]:
## define the hyperparameters to tune
param_grid = {
    'model__neurons':[16,32,64,128],
    'model__layer':[1,2,3],
    'epochs':[50,100]
}

In [19]:
#### Perform grid search
grid = GridSearchCV(estimator=model,param_grid=param_grid,n_jobs=-1,cv=3)
grid_result = grid.fit(X_train,y_train)

## Print the best hyperparameters and the best score
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

d:\Krish_Naik\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best: 0.857375 using {'epochs': 50, 'model__layer': 1, 'model__neurons': 128}
